# Tidy Data

In this notebook, you will learn a useful way to organize your data, a structure called tidy data. Getting your data into this format requires some upfront work, but that work pays off in the long term.

*Tidy datasets are all alike, but every messy dataset is messy in its own way.*

You can represent the same underlying data in multiple ways. The example below shows the same data organized in four different ways. Each dataset shows the same values of four variables country, year, population, and cases, but each dataset organizes the values in a different way.

In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
df1 = pd.DataFrame(
    [['Afghanistan',1999,745,19987071],
     ['Afghanistan',2000,2666,20595360],
     ['Brazil',1999,37737,172006362],
     ['Brazil',2000,80488,174504898],
     ['China',1999,212258,1272915272],
     ['China',2000,213766,1280428583]],
    columns=['country', 'year', 'cases', 'population'])
df1

In [ ]:
df2 = pd.DataFrame(
    [['Afghanistan',1999,'cases',745],
     ['Afghanistan',1999,'population',19987071],
     ['Afghanistan',2000,'cases',2666],
     ['Afghanistan',2000,'population',20595360],
     ['Brazil',1999,'cases',37737],
     ['Brazil',1999,'population',172006362],
     ['Brazil',2000,'cases',80488],
     ['Brazil',2000,'population',174504898],
     ['China',1999,'cases',212258],
     ['China',1999,'population',1272915272],
     ['China',2000,'cases',213766],
     ['China',2000,'population',1280428583]],
    columns=['country', 'year', 'type', 'count'])
df2

In [ ]:
df3 = pd.DataFrame(
    [['Afghanistan',1999,'745/19987071'],
     ['Afghanistan',2000,'2666/20595360'],
     ['Brazil',1999,'37737/172006362'],
     ['Brazil',2000,'80488/174504898'],
     ['China',1999,'212258/1272915272'],
     ['China',2000,'213766/1280428583']],
    columns=['country','year','rate'])
df3

In [ ]:
# cases
df4a = pd.DataFrame(
    [['Afghanistan',745,2666],
     ['Brazil',37737,80488],
     ['China',212258,213766]],
    columns=['country','1999','2000'])
df4a

In [ ]:
# population
df4b = pd.DataFrame(
    [['Afghanistan',19987071,20595360],
     ['Brazil',172006362,174504898],
     ['China',1272915272,1280428583]],
    columns=['country','1999','2000'])
df4b

These are all representations of the same underlying data, but they are not equally easy to use. The tidy dataset will be much easier to work with.

Given each of the four dataframes, try to write a script to answer the following questions.

- Find the number of cases in China in 2000

In [ ]:
# df1


In [ ]:
# df2


In [ ]:
# df3 


In [ ]:
# df4 (a or b)


- Find the total number of cases in all countries in 1999

In [ ]:
# df1


In [ ]:
# df2


In [ ]:
# df3 


In [ ]:
# df4 (a or b)


## What is a tidy dataset? 

There are three interrelated rules which make a dataset tidy:
- Each variable must have its own column.
- Each observation must have its own row.
- Each value must have its own cell.

In [ ]:
from IPython.display import Image
Image('https://d33wubrfki0l68.cloudfront.net/6f1ddb544fc5c69a2478e444ab8112fb0eea23f8/91adc/images/tidy-1.png')

Among the four dataframes above, which one is tidy? 

Tidy data makes a lot of operations easier. 

In [ ]:
# Creating a new column
# Compute rate per 10,000


In [ ]:
# Data summarization
# Compute cases per year


In [ ]:
# For each country, calculate the total number of cases and average population. 


## Reshaping Data

Tidy data is good. Unfortunately, in practice, most data that you will encounter will be untidy in one way or another. Hence, for most real analyses, you’ll need to do some tidying: 
1. Figure out what the variables and observations are. 
2. Resolve one of two common problems:
 - One variable might be spread across multiple columns.
 - One observation might be scattered across multiple rows.

### Melt: to gather columns into rows

A common problem is a dataset where some of the column names are not names of variables, but values of a variable. 

Take df4a: the column names 1999 and 2000 represent values of the year variable, the values in the 1999 and 2000 columns represent values of the cases variable, and each row represents two observations, not one.

In [ ]:
df4a

In [ ]:
# Use melt() to gather columns into rows
tidy4a = df4a.melt(id_vars=['country'], value_vars=['1999','2000'],
                   var_name='year', value_name='cases')
tidy4a

In [ ]:
# Similarly, apply melt() to df4b
tidy4b = df4b.melt(id_vars=['country'], value_vars=['1999','2000'],
                   var_name='year', value_name='population')
tidy4b

In [ ]:
# Join the two dataframes into one
tidy4 = pd.merge(tidy4a, tidy4b, on=['country','year'])
tidy4

### Pivot: to spread rows into columns

In contrast, another problem is that an observation is scattered across multiple rows. 

For example, take df2: an observation is a country in a year, but each observation is spread across two rows.

In [ ]:
df2

In [ ]:
df2.pivot(index=['country','year'], columns='type', values='count')

In [ ]:
df2.pivot(index=['country','year'], columns='type', values='count').reset_index()

In [ ]:
# Alternatively, you can use the pivot_table() method
df2.pivot_table(index=['country','year'], columns='type', values='count').reset_index()

### Split a column into columns

In [ ]:
df3

In [ ]:
df3[['cases','population']] = df3['rate'].str.split("/", expand=True).astype(int)
df3

In [ ]:
df3.info()

In [ ]:
df3.drop(columns=['rate'], inplace=True)
df3

### Missing values

Changing the representation of a dataset brings up an important subtlety of missing values. Surprisingly, a value can be missing in one of two possible ways:
- Explicitly, i.e. flagged with NaN.
- Implicitly, i.e. simply not present in the data.

In [ ]:
stocks = pd.DataFrame(
    [[2015,1,1.88],
     [2015,2,0.59],
     [2015,3,0.35],
     [2015,4,None],
     [2016,2,0.92],
     [2016,3,0.17],
     [2016,4,2.66]],
    columns=['year','qtr','return'])
stocks

There are two missing values in this dataset:
- The return for the fourth quarter of 2015 is explicitly missing, because the cell where its value should be instead contains NaN.
- The return for the first quarter of 2016 is implicitly missing, because it simply does not appear in the dataset.

**An explicit missing value is the presence of an absence; an implicit missing value is the absence of a presence.**

In [ ]:
# To reveal all missing values explicitly
stocks2 = stocks.pivot(index='qtr', columns='year', values='return').reset_index()
stocks2

In [ ]:
# Reshape the dataframe back to a tidy form
stocks2.melt(id_vars='qtr', value_vars=[2015,2016],
             var_name='year', value_name='return')

## Case Study

The dataset "who.csv" contains tuberculosis (TB) cases broken down by year, country, age, gender, and diagnosis method. The data comes from the 2014 World Health Organization Global Tuberculosis Report, available at http://www.who.int/tb/country/data/download/en/. 

In [ ]:
who = pd.read_csv('https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/data/who.csv', header=0)
who.head()

In [ ]:
who.shape

In [ ]:
who.columns

It is obvious that this dataset is quite messy. So let's tidy it up. 

The best place to start is almost always to gather together the columns that are not variables. Let’s have a look at what we’ve got:
- It looks like ``country``, ``iso2``, and ``iso3`` are three variables that redundantly specify the country.
- ``year`` is clearly also a variable.
- We don’t know what all the other columns are yet, but given the structure in the variable names (e.g. ``new_sp_m014``, ``new_ep_m014``, ``new_ep_f014``) these are likely to be values, not variables.

According to WHO's data dictionary:
1. The first three letters of each column denote whether the column contains ``new`` or ``old`` cases of TB. In this dataset, each column contains ``new`` cases.
2. The next 2~3 letters describe the type of TB:
 - ``rel`` stands for cases of relapse
 - ``ep`` stands for cases of extrapulmonary TB
 - ``sn`` stands for cases of pulmonary TB that could not be diagnosed by a pulmonary smear (smear negative)
 - ``sp`` stands for cases of pulmonary TB that could be diagnosed by a pulmonary smear (smear positive)
3. The sixth letter gives the sex of TB patients. The dataset groups cases by males (``m``) and females (``f``).
4. The remaining numbers give the age group. The dataset groups cases into seven age groups:
 - ``014`` = 0 – 14 years old
 - ``1524`` = 15 – 24 years old
 - ``2534`` = 25 – 34 years old
 - ``3544`` = 35 – 44 years old
 - ``4554`` = 45 – 54 years old
 - ``5564`` = 55 – 64 years old
 - ``65`` = 65 or older

Your final dataframe should contain the following columns: **country**, **year**, **cases**, **type**, **sex**, **age**

In [ ]:
# Tidy up the data
